# L4d: Production-Planning Shortest Path

Suppose we are planning a production process and have two ways to carry it out. One route uses five steps but includes several expensive operations. A second route uses six steps but replaces the costly operations with cheaper ones. Which route has the lower total cost, and could an equipment discount change our choice?

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * **Model the production process:** Build a weighted directed graph from the production edge list, connecting the available production steps and their costs to routes through the graph.
> * **Compute and validate the least-cost route:** Calculate both route costs by hand, compare the results with Dijkstra and Bellman–Ford, and reconstruct the selected route from its predecessors.
> * **Implement a break-even calculation:** Calculate the step cost at which the two routes have equal total cost and use this threshold to explain when a discount changes the preferred route.

We represent the process as a weighted directed graph: vertices describe process states, directed edges describe the steps connecting those states, and edge weights give the step costs. Adding the weights along a route gives its total cost. We use the shortest-path algorithms developed in [L4c](../L4c/CHEME-5800-L4c-Lecture-ShortestPathAlgorithms-Fall-2026.ipynb) to select the least-cost route and examine how that choice changes when one step becomes less expensive.

Let's get started!

___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines the data folder path, and loads the course package, student implementation, and plotting helper.

Let's set up our code environment:


In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

See the [Julia documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5800 course package documentation](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/) for the functions and types used here.

The local [`src/Compute.jl`](src/Compute.jl) file supplies the route-cost and route-reconstruction functions used in Tasks 2 and 3.

### Shared plotting setup

The following code cell defines the graph layout used in Tasks 2 and 3. Run it once before beginning the tasks.

We use the same vertex positions in both route figures to make changes easy to compare. Row `v` of `node_coordinates` gives the plot position of vertex `v`; `start_vertex` and `finish_vertex` select vertices 1 and 9 for the route calculations and figure highlights.

In [ ]:
# Define the route-figure layout: one (x, y) row per vertex, matching the Task 1 schematic.
node_coordinates = [
    10.0 10.0 ; # 1 start
    11.0 10.0 ; # 2
    11.0 11.0 ; # 3 upper route
    13.0 11.0 ; # 4 upper route
    13.0 10.0 ; # 5
    11.0  9.0 ; # 6 lower route
    12.0  9.0 ; # 7 lower route
    13.0  9.0 ; # 8 lower route
    14.0 10.0 ; # 9 completion
];
start_vertex = 1;  # every candidate route begins at vertex 1
finish_vertex = 9; # every candidate route ends at vertex 9

[The `plotroute(...)` function](docs/production-functions.md#plotroute) draws the graph and edge costs, highlights the selected route in red, and marks the start and finish vertices. The implementation is supplied in [`src/Visualization.jl`](src/Visualization.jl) and loaded by [`Include.jl`](Include.jl).

___

## Task 1: Build the production graph
In this task, we construct a graph model from the production edge list, then check that its connections and costs represent the two routes we want to compare.

The process begins at vertex 1 and ends at vertex 9. Both routes share steps `(1, 2)` and `(5, 9)`; they differ in how they move from vertex 2 to vertex 5.

<div>
    <center>
        <img src="figs/Fig-Branch-Schematic.svg" width="480" alt="Directed production graph: start vertex 1 leads to vertex 2, which splits into an upper route through vertices 3 and 4 and a lower route through vertices 6, 7, and 8; both routes rejoin at vertex 5 before the completion vertex 9"/>
    </center>
</div>

The process topology and costs are stored in [`data/Production-Process.edgelist`](data/Production-Process.edgelist), with one non-comment record per directed production step.

> **How is one production step encoded?**
>
> Each record contains three comma-separated fields: `source`, `target`, and `cost`. The first two identify the starting and ending vertices; `cost` gives the step’s expense in arbitrary cost units. Lines beginning with `#` are comments. For example, `3,4,8` describes a step from vertex 3 to vertex 4 with a cost of 8.

[The `MyGraphEdgeModels(...)` function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.MyGraphEdgeModels-Tuple%7BString%2C%20Function%7D) reads the file and passes each data record to the parser below, which extracts the vertex identifiers and step cost.

In [ ]:
"""
    edgerecordparser(record::String, delim::Char = ',') -> Tuple{Int64, Int64, Float64}

Parse one `source,target,cost` record into an `(Int64, Int64, Float64)` edge tuple.

### Arguments
- `record`: The edge record string to parse.
- `delim`: The delimiter used to split the record.

### Returns
- A tuple containing the source vertex id, target vertex id, and cost of the edge.

### Errors
- `ArgumentError`: The record does not have exactly three fields.
"""
function edgerecordparser(record::String, delim::Char = ',')

    fields = strip.(split(record, delim)); # remove whitespace around the fields
    length(fields) == 3 || throw(ArgumentError("expected source,target,cost but got: $(record)"))

    source = parse(Int64, fields[1]);  # source vertex id
    target = parse(Int64, fields[2]);  # target vertex id
    cost = parse(Float64, fields[3]);  # cost of completing the step

    return (source, target, cost)
end;

We locate the edge file using the data folder path defined in [`Include.jl`](Include.jl):

In [ ]:
path_to_edge_file = joinpath(CHEME5800_L4D_DATA, "Production-Process.edgelist"); # the graph shown in the schematic

[The `MyGraphEdgeModels(...)` function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.MyGraphEdgeModels-Tuple%7BString%2C%20Function%7D) returns a dictionary containing one edge model per production step. Its keys follow file order, starting at zero; the graph builder assigns separate edge IDs later.

In [ ]:
myedgemodels = MyGraphEdgeModels(path_to_edge_file, edgerecordparser, delim = ',', comment = '#')

[The `build(...)` function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.build-Union%7BTuple%7BT%7D%2C%20Tuple%7BType%7BT%7D%2C%20Dict%7BInt64%2C%20MyGraphEdgeModel%7D%7D%7D%20where%20T%3C%3AAbstractGraphModel) creates a [directed graph model](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.MySimpleDirectedGraphModel) containing the vertices, connections, and step costs needed by the shortest-path algorithms.

In [ ]:
directedgraphmodel = build(MySimpleDirectedGraphModel, myedgemodels);

### Check the graph model

Let’s inspect how `directedgraphmodel` stores the production steps and their costs. We use [the `typeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Core.typeof) to identify the model’s type and [the `fieldnames(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.fieldnames) to list its fields:

In [ ]:
typeof(directedgraphmodel) |> T -> fieldnames(T) # inspect the graph model's stored containers

The `edgesinverse` dictionary maps each edge ID to its `(source, target)` pair. These IDs start at one and differ from the parser keys. For example, edge 4 connects vertices 3 and 4. The tables below use these graph-model IDs.

In [ ]:
directedgraphmodel.edgesinverse

The `edges` dictionary maps each `(source, target)` pair to its step cost. For example, the record `3,4,8` becomes the entry `(3, 4) => 8.0`. We use these entries to calculate route costs and to change a step cost in Task 3.

In [ ]:
directedgraphmodel.edges

The table combines `edgesinverse` and `edges` to show each edge’s ID, endpoints, and cost:

In [ ]:
let
    edges = directedgraphmodel.edges;
    edgesinverse = directedgraphmodel.edgesinverse;
    df = DataFrame();
    for i in sort(collect(keys(edgesinverse)))
        (s, t) = edgesinverse[i];
        push!(df, (edge = i, s = s, t = t, cost = edges[(s, t)]));
    end
    pretty_table(df)
end

Check that the table contains nine directed steps and that their endpoints match the arrows in the schematic. The shared steps `(1, 2)` and `(5, 9)` each cost 1, so they contribute equally to both route costs.

Between vertices 2 and 5, the upper branch costs $4+8+2=14$, while the lower branch costs $2+2+2+2=8$. The lower route therefore costs 6 units less.

___

## Task 2: Compute the least-cost route

In this task, we calculate both route costs by hand, then use Dijkstra and Bellman–Ford to find the least-cost route and check their results against our calculations.

A route $\langle v_0,v_1,\ldots,v_k\rangle$ contains $k$ directed steps, each connecting consecutive vertices. If $w(v_i,v_{i+1})$ is the cost of step $(v_i,v_{i+1})$, the total route cost is given by:
$$
C=\sum_{i=0}^{k-1}w(v_i,v_{i+1}).
$$

Including the shared start and completion steps, the upper route costs $1+4+8+2+1=16$, while the lower route costs $1+2+2+2+2+1=10$. The lower route is therefore cheaper despite using six steps rather than five.

[The `L4dProductionPlanning.route_cost(...)` function](docs/production-functions.md#route_cost) sums the step costs along a route. It is already complete and rejects empty routes or vertex sequences containing a missing edge. We use it to calculate both totals:

In [ ]:
upper_route = [1, 2, 3, 4, 5, 9];    # fewer steps, two of them expensive
lower_route = [1, 2, 6, 7, 8, 5, 9]; # more steps, each of them cheap
upper_cost = L4dProductionPlanning.route_cost(directedgraphmodel.edges, upper_route);
lower_cost = L4dProductionPlanning.route_cost(directedgraphmodel.edges, lower_route);
(upper = upper_cost, lower = lower_cost)

### Find and reconstruct the least-cost route

[The `findshortestpath(...)` function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.findshortestpath-Union%7BTuple%7BT%7D%2C%20Tuple%7BT%2C%20MyGraphNodeModel%7D%7D%20where%20T%3C%3AAbstractGraphModel) takes the graph, starting-node model, and algorithm. It returns two dictionaries: `d[v]` gives the minimum cost of reaching vertex `v`, and `p[v]` records the preceding vertex on that route. The starting vertex has no predecessor.

For this process, `d[finish_vertex]` gives the least cost of reaching completion, while `p` records the predecessors needed to recover the production steps. We first use Dijkstra’s algorithm:

In [ ]:
(d, p) = let
    startnode = directedgraphmodel.nodes[start_vertex]; # the node model, not the id
    (d, p) = findshortestpath(directedgraphmodel, startnode, algorithm = DijkstraAlgorithm());
    (d, p)
end;

For example, `p[5] = 8` tells us that the selected route enters vertex 5 from vertex 8. [The `L4dProductionPlanning.reconstruct_route(...)` function](docs/production-functions.md#reconstruct_route) follows predecessors backward from vertex 9 to vertex 1, then reverses the sequence to return the production route from start to finish:

In [ ]:
shortest_route = L4dProductionPlanning.reconstruct_route(p, finish_vertex);
(distance = d[finish_vertex], route = shortest_route)

Dijkstra selects the lower route, $1\rightarrow2\rightarrow6\rightarrow7\rightarrow8\rightarrow5\rightarrow9$, with total cost 10. The figure highlights this route in red:

In [ ]:
plotroute(directedgraphmodel, shortest_route, node_coordinates)

### Verify the route with Bellman–Ford

We now solve the same graph with Bellman–Ford. Both algorithms should return the lower route with total cost 10:

In [ ]:
(d_bellman, p_bellman) = let
    startnode = directedgraphmodel.nodes[start_vertex];
    (d, p) = findshortestpath(directedgraphmodel, startnode, algorithm = BellmanFordAlgorithm());
    (d, p)
end;

Run the checks below to compare the route costs and selected vertex sequences with our hand calculations. They also verify that the supplied route-cost function assigns zero cost to a one-vertex route and rejects empty routes or steps absent from the graph.

In [ ]:
@testset "L4d least-cost route" begin
    # Check the route-cost function against the hand arithmetic.
    @test upper_cost == 16.0
    @test lower_cost == 10.0
    @test L4dProductionPlanning.route_cost(directedgraphmodel.edges, [start_vertex]) == 0.0
    @test_throws ArgumentError L4dProductionPlanning.route_cost(directedgraphmodel.edges, Int64[])
    @test_throws ArgumentError L4dProductionPlanning.route_cost(directedgraphmodel.edges, [1, 3])

    # Check Dijkstra's result against the hand calculation.
    @test d[finish_vertex] == lower_cost
    @test shortest_route == lower_route

    # Cross-check Dijkstra's result with Bellman–Ford.
    @test d_bellman[finish_vertex] == d[finish_vertex]
    @test L4dProductionPlanning.reconstruct_route(p_bellman, finish_vertex) == shortest_route
end;

The upper route costs 6 units more than the lower route. We next examine whether an equipment discount can make it cheaper.

___

## Task 3: Evaluate an equipment discount

In this task, we reduce the cost of step `(3, 4)` and recompute the least-cost route. We then derive and implement a calculation of the step cost at which the two routes have equal total cost.

We use [the `deepcopy(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.deepcopy) to create a graph we can modify while preserving the original costs for comparison:

In [ ]:
discounted_graphmodel = deepcopy(directedgraphmodel); # a copy we can edit; the baseline stays as it was

The `discount_factor` specifies the fraction of the original step cost that remains: `0.0` makes the step free, and `1.0` leaves it unchanged.

Using the baseline route totals, predict which route will be cheaper when step `(3, 4)` is free. We then set its cost to zero in the copied graph:

In [ ]:
let
    discount_factor = 0.0;           # fraction of the original price that remains: 0.0 is free, 1.0 is no discount
    discounted_steps = [(3, 4)];     # the (source, target) pairs that go on sale
    for step in discounted_steps
        discounted_graphmodel.edges[step] *= discount_factor;
    end
end;

The table compares the original and discounted costs. Only step `(3, 4)` changes, falling from 8 to 0:

In [ ]:
let
    edges = directedgraphmodel.edges;
    discounted_edges = discounted_graphmodel.edges;
    edgesinverse = directedgraphmodel.edgesinverse;
    df = DataFrame();
    for i in sort(collect(keys(edgesinverse)))
        (s, t) = edgesinverse[i];
        push!(df, (edge = i, s = s, t = t, cost = edges[(s, t)],
            discounted_cost = discounted_edges[(s, t)], Δ = discounted_edges[(s, t)] - edges[(s, t)]));
    end
    pretty_table(df)
end

We rerun Dijkstra on the discounted graph and reconstruct the least-cost route using the updated step cost:

In [ ]:
(d₁, p₁, discounted_route) = let
    startnode = discounted_graphmodel.nodes[start_vertex];
    (d, p) = findshortestpath(discounted_graphmodel, startnode, algorithm = DijkstraAlgorithm());
    (d, p, L4dProductionPlanning.reconstruct_route(p, finish_vertex))
end;

The figure highlights the recomputed route in red:

In [ ]:
plotroute(discounted_graphmodel, discounted_route, node_coordinates)

Setting step `(3, 4)` to zero lowers the upper-route cost from 16 to 8. The lower route still costs 10, so Dijkstra now selects the upper route.

### Find the break-even step cost

At what step cost do the routes tie? We find this break-even cost by setting their total costs equal.

Let $w$ be the new cost of step `(3, 4)`. The baseline route costs are $C_{\text{upper}}=16$ and $C_{\text{lower}}=10$, and the original step cost is $w_{34}=8$. Replacing that cost changes the upper total to $C_{\text{upper}}-w_{34}+w$. The lower total remains $C_{\text{lower}}$ because its route does not include the step. Equating the totals and solving for the break-even step cost $w^{\star}$ gives:
$$
\begin{aligned}
C_{\text{upper}} - w_{34} + w^{\star} &= C_{\text{lower}},\\
w^{\star} &= C_{\text{lower}} - \left(C_{\text{upper}} - w_{34}\right).
\end{aligned}
$$

For this process, $w^{\star}=10-(16-8)=2$. The routes tie when the step costs 2; the upper route is strictly cheaper below 2, and the lower route is strictly cheaper above 2. The derivation assumes that the discounted step appears exactly once on the candidate route and never on the reference route.

### Implement and check the break-even calculation

Implement this calculation in [the `breakeven_weight(...)` function](docs/production-functions.md#breakeven_weight) in [`src/Compute.jl`](src/Compute.jl), using the inputs and requirements below.

> __What must the break-even function return?__
>
> __Inputs__
>
> * `edges::AbstractDict`: the baseline `(source, target) => cost` dictionary.
> * `candidate_route::AbstractVector{<:Integer}`: the route that contains the discounted step exactly once.
> * `reference_route::AbstractVector{<:Integer}`: the route it competes against, which must not contain the step.
> * `step::Tuple{<:Integer, <:Integer}`: the `(source, target)` pair of the discounted step.
>
> __Output__
>
> * `Float64`: the cost of `step` at which the two routes cost the same. A negative value means no nonnegative price makes the candidate route cheaper.
>
> __Errors__
>
> * [`ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError): the step does not appear exactly once on the candidate route, the step appears on the reference route, or either route fails the route-cost checks.

Complete the two TODOs, save [`src/Compute.jl`](src/Compute.jl), and rerun the setup cell:

1. Check that the step appears exactly once on the candidate route and never on the reference route, raising an error otherwise.
2. Compute both route costs with [the `L4dProductionPlanning.route_cost(...)` function](docs/production-functions.md#route_cost) and return the break-even cost from the formula.

Use your completed function to calculate the break-even cost for step `(3, 4)`:

In [ ]:
breakeven_cost = L4dProductionPlanning.breakeven_weight(directedgraphmodel.edges, upper_route, lower_route, (3, 4))

We test step costs of 1.5 and 2.5 on separate graph copies to compare the selected routes below and above the break-even cost:

In [ ]:
let
    df = DataFrame();
    for price in (breakeven_cost - 0.5, breakeven_cost + 0.5)
        trial = deepcopy(directedgraphmodel);
        trial.edges[(3, 4)] = price;
        (d, p) = findshortestpath(trial, trial.nodes[start_vertex], algorithm = DijkstraAlgorithm());
        route = L4dProductionPlanning.reconstruct_route(p, finish_vertex);
        push!(df, (price = price, distance = d[finish_vertex], route = join(route, " → ")));
    end
    pretty_table(df)
end

Run the checks below to verify the copied graph, discounted route, and break-even result. They also check that your function rejects a step outside the candidate route or one shared by both routes.

In [ ]:
@testset "L4d discount scenario" begin
    # Check that the copy changed and the baseline did not.
    @test discounted_graphmodel.edges[(3, 4)] == 0.0
    @test directedgraphmodel.edges[(3, 4)] == 8.0

    # Check that the plan moves to the upper route.
    @test d₁[finish_vertex] == 8.0
    @test discounted_route == upper_route

    # Check the break-even contract.
    @test breakeven_cost == 2.0
    @test_throws ArgumentError L4dProductionPlanning.breakeven_weight(directedgraphmodel.edges, upper_route, lower_route, (6, 7))
    @test_throws ArgumentError L4dProductionPlanning.breakeven_weight(directedgraphmodel.edges, upper_route, lower_route, (1, 2))
end;

With the supplied baseline costs, the upper route is selected at 1.5, and the lower route at 2.5. The routes tie at 2, a 75 percent reduction from the original step cost of 8. The upper route becomes strictly cheaper with a larger reduction, provided all other costs remain fixed. If you change the baseline step costs, your break-even cost and route comparison may differ.

___


## Summary

We compared two production routes and calculated how a discount changes the least-cost plan.

> __Key Takeaways:__
>
> * **Production graph:** We used the production edge list to build a weighted directed graph of the available steps and their costs. Each route represented a production plan whose cost was the sum of its edge weights.
> * **Route selection:** We calculated route costs of 16 and 10 by hand, verified the lower route with Dijkstra and Bellman–Ford, and reconstructed its production steps from the predecessor map.
> * **Break-even analysis:** We derived and implemented a calculation to identify the step cost at which two production routes have the same total cost. We used this threshold to determine how large a cost reduction must be to change the preferred production plan.

In Week 5, we use maximum-flow models to ask how much material a network can carry, connecting network algorithms to linear programming.
